In [ ]:
import sys
sys.path.insert(0, "../src")
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

from askmydocs.loader import load_document
from askmydocs.splitter import split_documents
from askmydocs.embedder import embed_texts
from pathlib import Path

In [ ]:
pdf_path = Path("../data/uploads/RGPD.pdf")

pages = load_document(pdf_path)
chunks = split_documents(pages)
print(f"{len(chunks)} chunks chargés")

texts = [c["text"] for c in chunks]
pages_num = [c["page"] for c in chunks]

In [ ]:
print("Calcul des embeddings... (peut prendre 1-2 min)")
embeddings = np.array(embed_texts(texts))
print(f"Shape des embeddings : {embeddings.shape}")
# (nombre de chunks, 384)

In [ ]:
# PCA : on projette les 384 dimensions sur 2 pour pouvoir visualiser
pca = PCA(n_components=2)
coords = pca.fit_transform(embeddings)

plt.figure(figsize=(12, 8))
scatter = plt.scatter(
    coords[:, 0], coords[:, 1],
    c=pages_num, cmap="viridis",
    alpha=0.6, s=20
)
plt.colorbar(scatter, label="Numéro de page")
plt.title("Visualisation 2D des embeddings du RGPD (PCA)")
plt.xlabel("Composante principale 1")
plt.ylabel("Composante principale 2")
plt.tight_layout()
plt.savefig("../docs/embeddings_pca.png", dpi=150)  # pour le README
plt.show()

In [ ]:
from askmydocs.embedder import embed_query

question = "amendes administratives et sanctions"
q_vector = np.array(embed_query(question))
q_coords = pca.transform([q_vector])

plt.figure(figsize=(12, 8))
plt.scatter(coords[:, 0], coords[:, 1], c="lightgray", alpha=0.4, s=20, label="Chunks")
plt.scatter(q_coords[:, 0], q_coords[:, 1], c="red", s=200, marker="*",
            label=f"Question : '{question}'")
plt.title("Où se situe une question dans l'espace des embeddings")
plt.legend()
plt.tight_layout()
plt.show()